# Handoff notes
| File | Contents |
|---|---|
| `restaurants.csv` | one row per restaurant: `business_id`, `name`, `stars`, `address`, `city`, `state`, `postal_code`, `review_count`, `is_open`, `hours` (JSON string), `n_reviews_used`, `text` (cleaned reviews concatenated), and the label columns (`romantic`, `GoodForKids`, `GoodForGroups`, `LoudNoise`; NaN = missing) |
| `reviews.csv` | one row per review: `review_id`, `business_id`, `user_id`, `date`, `stars`, `useful`, `funny`, `cool`, `text_raw`, `text_clean` |
| `tips.csv` | one row per tip: `business_id`, `user_id`, `date`, `compliment_count`, `text_raw`, `text_clean` |
| `tfidf_uni.npz`, `tfidf_bi.npz` | sparse TF-IDF matrices, rows aligned with `restaurants.csv` |
| `vectorizer_*.joblib` | fitted vectorizers (for top-weighted words) |
| `label_summary.csv` | sample size and class balance per label |

# 1. Data & Prepocessing

**Pipeline:** load Yelp JSON → filter one city and restaurants with ≥10 reviews → clean and lemmatize text → concatenate up to 100 reviews per restaurant → pick occasion labels → build TF-IDF. Tips for the same restaurants are cleaned and saved separately.

**First run:** `pip install pandas numpy scipy scikit-learn joblib nltk`, download and unzip the Yelp Open Dataset and set `DATA_DIR` below.

In [32]:
import ast, json, os, re
from collections import Counter, defaultdict

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

DATA_DIR = "data"
CITY = "Philadelphia"
MIN_REVIEWS = 10
MAX_REVIEWS = 100
MIN_DF = 5                     # TF-IDF: ignore terms in fewer than this many restaurants
OUT_DIR = "out"

BUSINESS_FILE = "yelp_academic_dataset_business.json"
REVIEW_FILE = "yelp_academic_dataset_review.json"
TIP_FILE = "yelp_academic_dataset_tip.json"

# 2. Text cleaning helpers

In [33]:

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

STOP = set(stopwords.words("english"))

lemmatize = WordNetLemmatizer().lemmatize
LEMMA_BACKEND = "nltk WordNetLemmatizer"

URL_RE = re.compile(r"https?://\S+|www\.\S+")
NONALPHA_RE = re.compile(r"[^a-z\s]")   # removes digits and punctuation

def clean_text(text):
  t = URL_RE.sub(" ", str(text).lower())
  t = NONALPHA_RE.sub(" ", t)
  toks = [lemmatize(w) for w in t.split() if w not in STOP and len(w) > 1]
  return " ".join(toks)

clean_text("Loved the dishes! Visit https://x.com 5 times, the staff were amazing.")

'loved dish visit time staff amazing'

# 3. Attribute parsing
Yelp stores attributes as strings like `"u'quiet'"`, `'True'` or `"{'romantic': False, ...}"`, so they need parsing.
`NoiseLevel` is multi-class, so it is turned into a binary label (`LoudNoise` = loud or very loud).

In [34]:
def parse_attr(v):
    if v is None:
        return None
    if isinstance(v, (bool, dict)):
        return v
    s = str(v).strip()
    if s in ("None", ""):
        return None
    try:
        out = ast.literal_eval(s)
    except Exception:
        return s.strip("u'\"").lower()
    return out.lower() if isinstance(out, str) else out

def bool_label(v):
    v = parse_attr(v)
    return float(v) if isinstance(v, bool) else np.nan

def extract_labels(attrs):
    attrs = attrs or {}
    amb = parse_attr(attrs.get("Ambience"))
    amb = amb if isinstance(amb, dict) else {}
    noise = parse_attr(attrs.get("NoiseLevel"))
    return {
        "romantic": float(amb["romantic"]) if isinstance(amb.get("romantic"), bool) else np.nan,
        "GoodForKids": bool_label(attrs.get("GoodForKids")),
        "GoodForGroups": bool_label(attrs.get("RestaurantsGoodForGroups")),
        "LoudNoise": (np.nan if noise not in ("quiet", "average", "loud", "very_loud")
                      else float(noise in ("loud", "very_loud"))),
    }

def iter_json(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

## 3.1 Load restaurants and choose a city

In [35]:
def is_restaurant(b):
    cats = b.get("categories") or ""
    if isinstance(cats, list):
        return "Restaurants" in cats
    return "Restaurants" in cats

all_restaurants = [b for b in iter_json(os.path.join(DATA_DIR, BUSINESS_FILE)) if is_restaurant(b)]
print(len(all_restaurants), "restaurants in total")

city_counts = Counter((b["city"], b["state"]) for b in all_restaurants)
pd.DataFrame([(c, s, n) for (c, s), n in city_counts.most_common(15)],
             columns=["city", "state", "n_restaurants"])

52268 restaurants in total


,city,state,n_restaurants
0,Philadelphia,PA,5852
1,Tampa,FL,2960
2,Indianapolis,IN,2862
3,Nashville,TN,2502
4,Tucson,AZ,2466
5,New Orleans,LA,2259
6,Edmonton,AB,2166
7,Saint Louis,MO,1790
8,Reno,NV,1286
9,Boise,ID,847


## 3.2 Filter the chosen city and load reviews

In [36]:
assert CITY, "Set CITY in the config cell first"
biz = {b["business_id"]: b for b in all_restaurants if b["city"].lower() == CITY.lower()}
print(f"{len(biz)} restaurants in {CITY}")

reviews = defaultdict(list)
for r in iter_json(os.path.join(DATA_DIR, REVIEW_FILE)):
    if r["business_id"] in biz:
        reviews[r["business_id"]].append({
            "review_id": r["review_id"], "user_id": r["user_id"], "date": r["date"],
            "stars": r["stars"], "useful": r.get("useful", 0), "funny": r.get("funny", 0),
            "cool": r.get("cool", 0), "text": r["text"]})

keep = [b for b, rv in reviews.items() if len(rv) >= MIN_REVIEWS]
print(f"{len(keep)} restaurants with at least {MIN_REVIEWS} reviews")

tips_raw = [t for t in iter_json(os.path.join(DATA_DIR, TIP_FILE)) if t["business_id"] in set(keep)]
print(f"{len(tips_raw)} tips for the kept restaurants")

5854 restaurants in Philadelphia
4977 restaurants with at least 10 reviews
85866 tips for the kept restaurants


## 3.3 Clean text

Keep 100 most recent reviews per restaurant, concatenate

In [37]:
rest_rows, rev_rows = [], []
for bid in sorted(keep):
    rv = sorted(reviews[bid], key=lambda x: x["date"], reverse=True)[:MAX_REVIEWS]
    b = biz[bid]
    cleaned = [clean_text(r["text"]) for r in rv]
    rest_rows.append({
        "business_id": bid, "name": b["name"], "stars": b["stars"],
        "address": b.get("address"), "city": b["city"], "state": b.get("state"),
        "postal_code": b.get("postal_code", b.get("postal code")),   # docs spell it "postal code"
        "review_count": b.get("review_count"), "is_open": b.get("is_open"),
        "hours": json.dumps(b.get("hours")),
        "n_reviews_used": len(rv),
        "text": " ".join(cleaned),
        **extract_labels(b.get("attributes")),
    })
    for r, cl in zip(rv, cleaned):
        rev_rows.append({"review_id": r["review_id"], "business_id": bid, "user_id": r["user_id"],
                         "date": r["date"], "stars": r["stars"], "useful": r["useful"],
                         "funny": r["funny"], "cool": r["cool"],
                         "text_raw": r["text"], "text_clean": cl})

df = pd.DataFrame(rest_rows)
rev = pd.DataFrame(rev_rows)
tips = pd.DataFrame([{"business_id": t["business_id"], "user_id": t["user_id"], "date": t["date"],
                      "compliment_count": t.get("compliment_count", 0),
                      "text_raw": t["text"], "text_clean": clean_text(t["text"])} for t in tips_raw])
print(f"Business: {df.shape} | reviews: {rev.shape} | tips: {tips.shape}")
df.head()

Business: (4977, 16) | reviews: (290188, 10) | tips: (85866, 6)


,business_id,name,stars,address,city,state,postal_code,review_count,is_open,hours,n_reviews_used,text,romantic,GoodForKids,GoodForGroups,LoudNoise
0,-0M0b-XhtFagyLmsBtOe8w,Paris Wine Bar,3.5,2303 Fairmount Ave,Philadelphia,PA,19130,18,0,"{""Thursday"": ""17:0-0:0"", ""Friday"": ""17:0-0:0"",...",18,ok place extremely odd hole wall reminded rand...,0.0,0.0,0.0,0.0
1,-0PN_KFPtbnLQZEeb23XiA,Mr Wong's Chinese Restaurant,3.5,1849 Wolf St,Philadelphia,PA,19145,9,0,"{""Monday"": ""11:0-23:0"", ""Tuesday"": ""11:0-23:0""...",11,far away best chinese food order live point br...,0.0,1.0,0.0,0.0
2,-0TffRSXXIlBYVbb5AwfTg,IndeBlue Modern Indian Food & Spirits,4.5,205 South 13th St,Philadelphia,PA,19107,1097,1,"{""Monday"": ""0:0-0:0"", ""Tuesday"": ""16:0-22:0"", ...",100,went indieblue year ago happy hour found prett...,0.0,1.0,1.0,0.0
3,-0eUa8TsXFFy0FCxHYmrjg,Waterfront Gourmet Cafe & Deli,4.0,3131 Walnut St,Philadelphia,PA,19104,26,0,"{""Monday"": ""7:0-20:0"", ""Tuesday"": ""7:0-20:0"", ...",28,sitting icu waiting room several day hungry ti...,0.0,1.0,1.0,0.0
4,-1B9pP_CrRBJYPICE5WbRA,Spice 28,4.0,1228 Chestnut St,Philadelphia,PA,19107,822,1,"{""Monday"": ""11:0-22:0"", ""Tuesday"": ""11:0-22:30...",100,food absolutely amazing get food delivered giv...,0.0,1.0,1.0,0.0


## 3.4 Label summary

In [38]:
label_cols = ["romantic", "GoodForKids", "GoodForGroups", "LoudNoise"]
summ = pd.DataFrame([{
    "label": c,
    "n_labeled": int(df[c].notna().sum()),
    "n_missing": int(df[c].isna().sum()),
    "n_positive": int(df[c].sum()),
    "positive_rate": round(float(df[c].mean()), 3),
} for c in label_cols])
summ

,label,n_labeled,n_missing,n_positive,positive_rate
0,romantic,3869,1108,112,0.029
1,GoodForKids,4061,916,3028,0.746
2,GoodForGroups,4156,821,3298,0.794
3,LoudNoise,3614,1363,413,0.114


In [39]:
unique_summary = pd.DataFrame([{
    "label": c,
    "n_unique": df[c].nunique(),
    "unique_values": df[c].unique().tolist()
} for c in label_cols])

unique_summary

,label,n_unique,unique_values
0,romantic,2,"[0.0, nan, 1.0]"
1,GoodForKids,2,"[0.0, 1.0, nan]"
2,GoodForGroups,2,"[0.0, 1.0, nan]"
3,LoudNoise,2,"[0.0, nan, 1.0]"


In [40]:
SELECTED_LABELS = label_cols
df["missing_any_selected"] = df[SELECTED_LABELS].isna().any(axis=1)

print(f"Missing any: {df["missing_any_selected"].sum()} | Not missing any: {(~df["missing_any_selected"]).sum()}")

Missing any: 1797 | Not missing any: 3180


# 4. TF-IDF features

In [42]:
os.makedirs(OUT_DIR, exist_ok=True)
for name, ngr in (("uni", (1, 1)), ("bi", (1, 2)), ("tri", (1, 3))):
    vec = TfidfVectorizer(ngram_range=ngr, min_df=MIN_DF, max_df=0.8, sublinear_tf=True)
    X = vec.fit_transform(df["text"])
    sp.save_npz(os.path.join(OUT_DIR, f"tfidf_{name}.npz"), X)
    joblib.dump(vec, os.path.join(OUT_DIR, f"vectorizer_{name}.joblib"))
    print(f"tfidf_{name}: {X.shape}")

tfidf_uni: (4977, 26115)
tfidf_bi: (4977, 427994)
tfidf_tri: (4977, 592798)


# 5. Save to file

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "restaurants.csv"), index=False)
rev.to_csv(os.path.join(OUT_DIR, "reviews.csv"), index=False)
tips.to_csv(os.path.join(OUT_DIR, "tips.csv"), index=False)
summ.to_csv(os.path.join(OUT_DIR, "label_summary.csv"), index=False)
print("Saved to", OUT_DIR, os.listdir(OUT_DIR))

Saved to out ['vectorizer_uni.joblib', 'reviews.csv', 'label_summary.csv', 'tips.csv', 'vectorizer_bi.joblib', 'tfidf_bi.npz', 'tfidf_uni.npz', 'restaurants.csv']
